# 数组互操作与类型标注

学习目标：为数组输入添加类型标注和运行时检查，判断转换是否复制，并核对缓冲区共享和只读条件。

前置知识：Python 类型标注、数组缓冲区、视图与副本。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章为按需选读专题。示例只使用本地 CPU 数组；后续单元沿用首次导入的 np 和 npt。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 统一数组输入

收到三次测量的两列读数后，先把嵌套列表转成浮点数组，再计算每列均值。轴 0 表示测量次数，轴 1 表示两个通道，输入形状为 (3, 2)。

np.asarray 接受可转换的输入，并按 dtype 指定元素类型。它可能复用已有数组，也可能分配新数组；转换成功不等于没有复制。

In [1]:
import numpy as np
import numpy.typing as npt

incoming = [[10, 20], [12, 24], [14, 28]]
readings = np.asarray(incoming, dtype=np.float64)
print(readings.shape, readings.dtype)  # (3, 2) float64
print(readings.mean(axis=0))  # [12. 24.]：每个通道的三次测量均值

(3, 2) float64
[12. 24.]


## 2 类型标注与运行时检查

numpy.typing 提供数组相关的标注。下表的 npt 是 numpy.typing 的导入别名。

| 名称 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| npt.ArrayLike | 可转换为数组的输入类型 | 描述接受标量、序列、数组等输入的参数 |
| npt.NDArray | ndarray 类型别名 | 例如 npt.NDArray[np.float64] 描述 float64 元素类型，不限定具体形状 |
| npt.DTypeLike | 可转换为 dtype 的类型描述 | 描述 np.float32、"float32" 或 dtype 对象等参数 |

Python 不会自动执行这些标注的约束。下面的函数先转换 dtype，再检查二维、两列和至少一行；这些都是本例规定的输入条件。ArrayLike 也不保证输入一定能转换成所需的数值类型。

In [2]:
def as_readings(data: npt.ArrayLike) -> npt.NDArray[np.float64]:
    values = np.asarray(data, dtype=np.float64)
    if values.ndim != 2 or values.shape[1] != 2 or values.shape[0] == 0:
        raise ValueError("读数必须是至少一行、恰好两列的二维数组")
    return values


result = as_readings([[10, 20], [12, 24], [14, 28]])
print(result.shape, result.dtype)  # (3, 2) float64
print(result.mean(axis=0))  # [12. 24.]

(3, 2) float64
[12. 24.]


沿用 as_readings，比较一维输入、列数不符和空行输入。即使元素 dtype 正确，这些形状也不满足函数的约定。

In [3]:
# 预期 ValueError：输入是一维数组，不满足二维读数表的约定。
as_readings(np.array([1.0, 2.0]))

ValueError: 读数必须是至少一行、恰好两列的二维数组

In [4]:
# 预期 ValueError：读数表约定两列，这个输入有三列。
as_readings(np.ones((2, 3)))

ValueError: 读数必须是至少一行、恰好两列的二维数组

In [5]:
# 预期 ValueError：读数表约定至少一行，这个输入没有数据行。
as_readings(np.empty((0, 2)))

ValueError: 读数必须是至少一行、恰好两列的二维数组

DTypeLike 描述 dtype 参数可以采用哪些写法，实际转换仍由 np.dtype 或 np.asarray 完成。这里用字符串指定 float32；若改成 np.float32，含义相同。类型检查工具可分析标注，但运行这些单元本身不等于执行了静态类型检查。

In [6]:
dtype_spec: npt.DTypeLike = "float32"
resolved_dtype = np.dtype(dtype_spec)
values = np.asarray([1, 2, 3], dtype=resolved_dtype)
print(values, values.shape, values.dtype)  # [1. 2. 3.] (3,) float32
print(resolved_dtype == np.dtype(np.float32))  # True：两种描述得到同一 dtype

[1. 2. 3.] (3,) float32
True


## 3 asarray 的复制条件

NumPy 2 的 np.asarray 支持三种 copy 取值。dtype 和 order 指定的类型、内存排列也会影响能否复用输入。

| 写法 | 中文名称／含义 |
| --- | --- |
| copy=None | 默认策略：仅在需要时复制，例如列表转换、类型转换或排列改变 |
| copy=True | 创建副本 |
| copy=False | 禁止复制；无法满足条件时抛出 ValueError |

输入已经是类型和排列匹配的普通 ndarray 时，默认转换可以返回原对象。用 np.shares_memory 检查两个小数组是否共享存储；值相等不足以判断是否复制。

In [7]:
source = np.array([1.0, 2.0, 3.0], dtype=np.float64)
reused = np.asarray(source)
copied = np.asarray(source, copy=True)
converted = np.asarray(source, dtype=np.float32)

print(reused is source, np.shares_memory(source, reused))  # True True
print(np.shares_memory(source, copied), copied.dtype)  # False float64
print(np.shares_memory(source, converted), converted.dtype)  # False float32
copied[0] = 99
print(source, copied)  # 原数组仍为 [1. 2. 3.]；副本首项为 99

True True
False float64
False float32
[1. 2. 3.] [99.  2.  3.]


copy=False 是严格要求，不能把它理解为“尽量不复制”。把 Python 列表变为数组需要新存储，改变已有数组的元素类型通常也需要复制。

In [8]:
# 预期 ValueError：从 Python 列表创建数组需要分配数据，无法保证不复制。
np.asarray([1, 2, 3], dtype=np.float64, copy=False)

ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [9]:
source = np.array([1, 2, 3], dtype=np.int16)
# 预期 ValueError：int16 转为 float64 需要新存储，copy=False 禁止复制。
np.asarray(source, dtype=np.float64, copy=False)

ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

有步长的切片可以与原数组共享数据，但不一定满足接收方要求的连续排列。下面每隔一个元素取值；order="C" 要求按 C 顺序连续存储，因而需要复制。

In [10]:
source = np.arange(6, dtype=np.int16)[::2]
print(source, source.shape, source.dtype)  # [0 2 4] (3,) int16
print(source.flags.c_contiguous)  # False

# 预期 ValueError：跨步切片不是 C 连续数组，满足 order="C" 需要复制，但 copy=False 禁止复制。
np.asarray(source, order="C", copy=False)

[0 2 4] (3,) int16
False


ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [11]:
contiguous = np.asarray(source, order="C")
print(contiguous.flags.c_contiguous, np.shares_memory(source, contiguous))  # True False

True False


## 4 从缓冲区解释数值

缓冲区协议允许对象暴露底层字节。np.frombuffer 把这段字节解释成一维数组，通常建立视图，不逐项转换数值。

下面模拟四个小端、无符号 16 位整数，每项占两个字节。dtype="<u2" 中，< 表示小端，u 表示无符号整数，2 表示两个字节。字节本身不说明 dtype、字节序或多维形状，接收方必须按约定解释。

In [12]:
payload = bytearray([1, 0, 2, 0, 3, 0, 4, 0])
values = np.frombuffer(payload, dtype="<u2")
print(values, values.shape, values.dtype)  # [1 2 3 4] (4,) uint16（本机为小端）
print(values.flags.owndata, values.flags.writeable)  # False True：借用可写字节缓冲区

values[1] = 500
print(list(payload[:4]))  # [1, 0, 244, 1]：数组写入改变原缓冲区
payload[0] = 9
print(values)  # [9 500 3 4]：缓冲区写入也能从数组观察到
print(np.shares_memory(values, np.frombuffer(payload, dtype=np.uint8)))  # True

[1 2 3 4] (4,) uint16
False True
[1, 0, 244, 1]
[  9 500   3   4]
True


offset 按字节计算，count 按元素计算。若读取全部剩余字节，其长度必须是元素字节数的整数倍；否则无法组成完整元素。

In [13]:
payload = bytes([1, 0, 2, 0, 3, 0, 4, 0])
middle = np.frombuffer(payload, dtype="<u2", offset=2, count=2)
print(middle, middle.shape, middle.dtype)  # [2 3] (2,) uint16

# 预期 ValueError：uint16 每项占 2 字节，3 字节缓冲区不能组成完整元素。
np.frombuffer(bytes([1, 0, 2]), dtype="<u2")

[2 3] (2,) uint16


ValueError: buffer size must be a multiple of element size

## 5 只读权限与所有权

bytes 不可变，借用它的数组是只读的。需要独立修改时显式复制；不要假定换一个数组变量名就获得了可写副本。

In [14]:
payload = bytes([1, 0, 2, 0, 3, 0])
readonly = np.frombuffer(payload, dtype="<u2")
print(readonly.flags.writeable)  # False

# 预期 ValueError：视图共享不可变 bytes 的存储，不能写入。
readonly[0] = 8

False


ValueError: assignment destination is read-only

In [15]:
editable = readonly.copy()
editable[0] = 8
print(editable, readonly)  # [8 2 3] 与 [1 2 3]
print(editable.flags.writeable, np.shares_memory(editable, readonly))  # True False

[8 2 3] [1 2 3]
True False


共享还要求底层存储在使用期间有效。Python 缓冲区协议通过对象引用维持普通导出对象的生命周期；下面的 frombuffer 视图持有对 bytearray 存储的引用，删除原变量名不会立即销毁这段存储。

flags.owndata 表示数组是否拥有数据，base 可用于观察其依赖对象；判断两个数组的共享关系仍应检查 np.shares_memory。手工提供裸地址或接入外部资源时，必须另外遵守对方的生命周期约定，不能由这个 bytearray 例子推断任意指针都安全。

In [16]:
payload = bytearray([10, 20, 30])
borrowed = np.frombuffer(payload, dtype=np.uint8)
print(borrowed.flags.owndata, borrowed.base is not None)  # False True
del payload
borrowed[0] = 99
print(borrowed, borrowed.shape, borrowed.dtype)  # [99 20 30] (3,) uint8：存储仍有效

False True
[99 20 30] (3,) uint8


## 6 选学：数组转换与运算协议
不同库可以提供协议，让 NumPy 接收其数据，或者把运算交回原对象处理。强制转换成 ndarray 可能丢失原库的元数据和专有行为，应先明确接收方需要哪一种能力。

| 名称 | 中文名称／含义 | 用途 |
| --- | --- | --- |
| \_\_array\_interface\_\_ | 数组接口 | 用字典描述 shape、元素类型、地址和 strides，供接收方解释存储 |
| \_\_array\_\_ | 数组转换方法 | 返回 ndarray；NumPy 2 的实现应接收 dtype 和 copy 参数 |
| \_\_array\_ufunc\_\_ | 通用函数分派方法 | 自定义对象接管 np.add 等 ufunc 运算 |
| \_\_array\_function\_\_ | NumPy 函数分派方法 | 自定义对象接管支持该协议的高层函数，如 np.mean |

数组接口中的 data 除地址外还包含只读标记。这里仅检查形状、步长和权限，不手工修改地址。

In [17]:
source = np.arange(6, dtype=np.int16).reshape(2, 3)[:, ::2]
interface = source.__array_interface__
print(source)  # [[0 2], [3 5]]：保留第 0、2 列
print(interface["shape"], interface["strides"])  # (2, 2) (6, 4)，步长单位为字节
print(interface["typestr"], interface["data"][1])  # 本机 <i2 False：小端 int16，可写

[[0 2]
 [3 5]]
(2, 2) (6, 4)
<i2 False


下面的最小包装类只演示 \_\_array\_\_ 转换。它把 dtype 和 copy 要求交给 np.asarray：copy=True 必须返回副本；copy=False 无法满足时必须报错。它没有实现运算分派，不能据此认为任何包装对象都自动支持 NumPy 运算。

In [18]:
class Readings:
    def __init__(self, values: npt.ArrayLike):
        self.values = np.asarray(values)

    def __array__(self, dtype=None, copy=None):
        return np.asarray(self.values, dtype=dtype, copy=copy)


wrapped = Readings(np.array([1, 2, 3], dtype=np.int16))
shared = np.asarray(wrapped, copy=False)
separate = np.asarray(wrapped, dtype=np.float64, copy=True)
print(shared.shape, shared.dtype, np.shares_memory(shared, wrapped.values))  # (3,) int16 True
print(separate, separate.dtype)  # [1. 2. 3.] float64
print(np.shares_memory(separate, wrapped.values))  # False

(3,) int16 True
[1. 2. 3.] float64
False


## 7 选学：DLPack 交换
DLPack 是数组库交换张量数据的协议。np.from_dlpack 显式请求导出对象的 \_\_dlpack\_\_ 和 \_\_dlpack\_device\_\_ 接口；它不等同于 np.asarray 的隐式转换。

NumPy 的目标设备是 CPU。设备不同、dtype 不支持或存储表示不兼容时，零复制可能无法完成。NumPy 2.5 中 copy=False 禁止复制，无法满足时抛出 BufferError；copy=True 请求副本，导出方也需要支持该请求。

下面仅用 NumPy 自身作为本地 CPU 导出方。共享结果的可写性取决于导出方和协议支持，使用前应检查，不能笼统认为 DLPack 结果总是只读。

In [19]:
source = np.array([1.0, 2.0, 3.0], dtype=np.float64)
shared = np.from_dlpack(source, copy=False)
separate = np.from_dlpack(source, copy=True)
print(shared.shape, shared.dtype)  # (3,) float64
print(np.shares_memory(source, shared), np.shares_memory(source, separate))  # True False
print(shared.flags.writeable)  # 本次 NumPy 2.5.3 导入为 True
source[0] = 9
print(shared, separate)  # [9. 2. 3.] 与 [1. 2. 3.]

(3,) float64
True False
True
[9. 2. 3.] [1. 2. 3.]


对照只读导出数组和不支持的 Unicode dtype。NumPy 2.5 已把 DLPack 不支持类型等情况下的异常统一为 BufferError，不应沿用旧版本的 RuntimeError 异常类型。

In [20]:
source = np.array([1.0, 2.0, 3.0])
source.setflags(write=False)
shared = np.from_dlpack(source, copy=False)
print(shared.flags.writeable, np.shares_memory(source, shared))  # 本次为 False True

# 预期 BufferError：这个 Unicode 字符串数组的 dtype 不受 NumPy 的 DLPack 导出支持。
np.from_dlpack(np.array(["A", "B"]))

False True


BufferError: DLPack only supports signed/unsigned integers, float and complex dtypes (or dtypes registered by third-party packages).

## 8 选学：Array API 与迁移入口
Array API 规定一组跨数组库的公共接口。NumPy 2.3 起主命名空间以及 fft、linalg 命名空间兼容 2024.12 版标准；旧的 numpy.array_api 实验子模块已在 NumPy 2.0 移除。

数组的 \_\_array\_namespace\_\_ 方法返回相应的函数命名空间。希望跨库复用时，应使用标准规定的功能子集，并核对具体库支持的设备与版本。下面只展示 NumPy 后端，不构成跨库兼容性验证。

In [21]:
values = np.asarray([1.0, 2.0, 3.0])
xp = values.__array_namespace__(api_version="2024.12")
print(xp is np)  # True：本例获得 NumPy 命名空间
print(xp.sum(values))  # 6.0

True
6.0


维护旧 NumPy 代码时，先核对迁移指南中与接口有关的变化。

（1）copy=False 在 NumPy 2 中表示禁止复制；原意若只是“需要时才复制”，用 np.asarray 的默认策略。

（2）标量类型提升规则改变。与 Python 浮点数运算时，float32 可以继续保持 float32；明确要求更高精度时应显式选择类型。

（3）默认整数在 64 位系统上为 64 位，与 np.intp 对应；不能把旧 Windows 默认整数宽度当作跨平台约定。

（4）命名空间和部分方法有迁移，例如使用 np.ptp 代替已移除的 ndarray.ptp。涉及编译扩展时，还须处理 NumPy 2 的二进制兼容性变化。

In [22]:
low_precision = np.float32(3) + 3.0
explicit_precision = np.float32(3) + np.float64(3)
print(low_precision.dtype, explicit_precision.dtype)  # float32 float64
values = np.array([2, 5, 9], dtype=np.int64)
print(np.ptp(values))  # 7：使用公开函数求极差

float32 float64
7


## 9 选学：底层扩展入口
当已有 C 或 Fortran 数值程序需要接入数组时，再考虑编译扩展。下面只说明入口，不在本章构建工具链。

| 名称 | 中文名称／含义 | 适用任务 |
| --- | --- | --- |
| NumPy C-API | NumPy 的 C 接口 | 在扩展中创建、访问数组，并处理 dtype、步长、标志和内存引用 |
| ufunc C-API | 通用函数的 C 接口 | 为编译实现提供与 NumPy 通用函数衔接的入口 |
| F2PY | Fortran 到 Python 的接口生成工具 | 为已有 Fortran 程序生成可从 Python 调用的扩展 |

这些接口不能消除所有权、设备和存储表示限制。接入外部存储时，应明确由谁维持存储有效、是否允许写入、何时需要复制；编译扩展还需核对其 NumPy 版本兼容要求。

## 本章小结

（1）ArrayLike、NDArray 和 DTypeLike 表达接口类型；形状、取值和业务约束仍由运行时检查落实。

（2）np.asarray 是否复制取决于输入、dtype、排列和 copy。copy=False 不能满足时会失败。

（3）frombuffer 借用字节存储；必须约定类型和字节序，同时检查共享、权限及生命周期。需要隔离修改时创建副本。

（4）数组转换、运算分派、DLPack 和 Array API 服务于不同的互操作任务。零复制始终受设备和存储表示等条件约束。

## 练习

（1）为三列传感器输入编写带类型标注的转换函数：输出 float64，要求二维、三列且至少一行。分别检查有效输入、两列输入和零行输入；解释为什么只标注 NDArray[np.float64] 还不够。

In [23]:
valid = [[1, 2, 3], [4, 5, 6]]
wrong_columns = [[1, 2], [3, 4]]
empty = np.empty((0, 3))
# 在此编写函数并打印有效输入的 shape、dtype。
# 分别运行两个无效输入，解释 ValueError，并用注释说明标注与检查的分工。
# 检查标准：有效输出为 (2, 3)、float64，两个无效形状均被拒绝。

（2）先预测 a、b 和 c 的最终内容与共享关系，再运行核对。随后把目标改为“修改 b 不能改变 a”，只改转换语句并说明原因。

In [24]:
a = np.array([1, 2, 3], dtype=np.int16)
b = np.asarray(a)
c = np.asarray(a, dtype=np.float64)
b[0] = 7
# 在此运行前用注释写出三组预测值与两组共享关系。
print(a, b, c)
print(np.shares_memory(a, b), np.shares_memory(a, c))
# 在此按新的隔离修改要求改写 b 的转换方式，再重新运行本单元。

[7 2 3] [7 2 3] [1. 2. 3.]
True False


（3）接收到四个小端无符号 16 位整数的字节缓冲区。第一次要求“修改数组同步修改缓冲区”；第二次要求“修改数组必须保留收到的字节”。分别选择转换方式，修改首项后打印双方，并解释选择理由。

In [25]:
shared_payload = bytearray([1, 0, 2, 0, 3, 0, 4, 0])
isolated_payload = bytearray([1, 0, 2, 0, 3, 0, 4, 0])
# 在此两种要求分别转换并把首项改为 9。
# 检查标准：两份数组都是 (4,)、uint16；前者同步变更，后者保留原字节。
# 在此打印 flags.writeable；用注释解释是否需要 copy。

（4）选学：将一个本地 CPU 数值数组通过 DLPack 转换，检查值、形状、dtype、共享关系和写入权限。若接收端增加“不允许改变导出端数据”的要求，应怎样改变转换策略？说明这个实验为何不能证明任意设备都支持零复制。

In [26]:
exported = np.array([2.0, 4.0, 6.0], dtype=np.float32)
# 在此完成共享导入和满足隔离要求的导入，分别打印检查结果。
# 检查标准：两者值相同、形状为 (3,)、dtype 为 float32，共享关系应有区别。
# 在此用注释说明设备、类型表示与所有权的条件。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（numpy.org） | NumPy 2.5：[Typing](https://numpy.org/doc/2.5/reference/typing.html) 的 ArrayLike、NDArray、DTypeLike；[asarray](https://numpy.org/doc/2.5/reference/generated/numpy.asarray.html) 的 dtype、order、copy；[frombuffer](https://numpy.org/doc/2.5/reference/generated/numpy.frombuffer.html) 的 count、offset 与 Notes；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html)、[ndarray.flags](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.flags.html)、[ndarray.base](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.base.html)；[Interoperability](https://numpy.org/doc/2.5/user/basics.interoperability.html) 的数组转换及运算分派协议；[Array interface](https://numpy.org/doc/2.5/reference/arrays.interface.html) 的字典字段；[from_dlpack](https://numpy.org/doc/2.5/reference/generated/numpy.from_dlpack.html) 的设备和复制参数；[2.5 发布说明](https://numpy.org/doc/2.5/release/2.5.0-notes.html) 的 from_dlpack raises BufferError instead of RuntimeError；[Array API compatibility](https://numpy.org/doc/2.5/reference/array_api.html) 的标准版本与历史；[NumPy 2 迁移指南](https://numpy.org/doc/2.5/numpy_2_0_migration_guide.html) 的类型提升、默认整数、copy、命名空间及二进制兼容性；[C-API](https://numpy.org/doc/2.5/reference/c-api/index.html) 的数组与 ufunc 接口入口；[F2PY](https://numpy.org/doc/2.5/f2py/index.html) 的用途说明。 |
| Python 官方文档（docs.python.org） | Python 3.12：[typing](https://docs.python.org/3.12/library/typing.html) 开头关于运行时不强制执行标注的说明；[Built-in Types](https://docs.python.org/3.12/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview) 的 bytes、bytearray 与 memoryview；[Buffer Protocol](https://docs.python.org/3.12/c-api/buffer.html#buffer-structure) 的 obj 引用、只读标记与布局条件。 |
| Python Data API 标准（data-apis.org） | 2024.12：[数组命名空间方法](https://data-apis.org/array-api/2024.12/API_specification/generated/array_api.array.__array_namespace__.html) 的 api_version 和返回对象约定。 |